In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv('/home/sergi_alcala/sergi_data/CLEAN_AZTEC_Extension/citys/Paris.csv')

In [3]:
data.drop('date_time', axis=1, inplace=True)

In [4]:
for column in data.columns:
    np.save(f'/home/sergi_alcala/sergi_data/CLEAN_AZTEC_Extension/Benchmark/conext/AZTEC_DATA_PARIS/{column}.npy', data[column].values)

In [5]:
import sys, os, re
import numpy as np
import math
import time
import json
import pickle
from tqdm import tqdm
from matplotlib import pyplot as plt
from datetime import date


from networkx.readwrite import json_graph
import networkx as nx

sys.path.append('/home/sergi_alcala/sergi_data/CLEAN_AZTEC_Extension/Benchmark/conext/dashboard')
sys.path.append('/home/sergi_alcala/sergi_data/CLEAN_AZTEC_Extension/Benchmark/conext/orchestrator/')
sys.path.append('/home/sergi_alcala/sergi_data/CLEAN_AZTEC_Extension/Benchmark/conext/optimizer/')
sys.path.append('/home/sergi_alcala/sergi_data/CLEAN_AZTEC_Extension/Benchmark/conext/helpers/')
sys.path.append('/home/sergi_alcala/sergi_data/CLEAN_AZTEC_Extension/Benchmark/conext/structures/')


In [7]:
from generate_topology import generate_onelink_topology
import generate_tenants
import orchestrator
import tenant_predictor_only_pred
from helpers import *
import main

In [8]:
def purge(dir, pattern):
    for f in os.listdir(dir):
        if re.search(pattern, f):
            os.remove(os.path.join(dir, f))
def clean():
    print("Cleaning up...")
    # purge("/tmp/", "enso_tenants_state.json")
    # purge("/tmp/", "enso_tenants_dashboard.json")    

def usage():
    print("""\
        Usage:
run(toponame, n_tenants, penalty, type (a value per tenant), lambda (a value per tenant between 0 and 1), sigma (a value per tenant between 0 and 1)
type 1: eMBB (LAMBDA = 50, latency = 20ms, beta = 0, R = 1)
type 2: mMTC (LAMBDA = 10, sigma = 0, latency = 20ms, beta = 1, R = 1)
type 3: uRLCC (LAMBDA = 25, latency = 5ms, beta = 0.25, R = 1)
Exiting...""")

In [ ]:
T_mno = 1
Ctot = 1
today_date = date.today().strftime("%Y_%m_%d")

In [9]:
tenant_1 = np.load('/home/sergi_alcala/sergi_data/CLEAN_AZTEC_Extension/Benchmark/conext/AZTEC_DATA_PARIS/Google.npy')

In [10]:
tenant_1.shape

(17911,)

In [16]:
train_samples, val_samples = tenant_1[:int(0.7*len(tenant_1))], tenant_1[int(0.7*len(tenant_1)): int(0.9*len(tenant_1))]
test_samples = tenant_1[int(0.9*len(tenant_1)):]
train_samples.shape, val_samples.shape, test_samples.shape

((12537,), (3582,), (1792,))

In [15]:
train_samples.shape, val_samples.shape

((14328,), (3583,))

In [ ]:
nTenants        = 5
monetary_factor = 1

mon_samples     = 24*60*2
T_Slice         = 30
dur_slice       = T_Slice/T_mno
lengthHistory   = mon_samples

train_samples, val_samples = 40320, 20160 #, htest_samples  = 10080 - 480 - 120
start_test    = train_samples + val_samples + 480

idx_first     = 0
idx_last      = idx_first + T_Slice_interval_run*T_mno 

leasing_cost = 1